In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
import joblib
from pathlib import Path

In [2]:
base_path = Path("C:/Users/acer/Desktop/Vangaurd")

data = pd.read_csv(base_path / "data/processed/processed_train_FD001.csv")

In [3]:
print(data.columns.tolist())

['op_settings1', 'op_settings2', 'op_settings3', 'sensor2', 'sensor3', 'sensor4', 'sensor7', 'sensor8', 'sensor9', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor17', 'sensor20', 'sensor21', 'raw_RUL', 'sensor2_mean', 'sensor2_std', 'sensor3_mean', 'sensor3_std', 'sensor4_mean', 'sensor4_std', 'sensor7_mean', 'sensor7_std', 'sensor8_mean', 'sensor8_std', 'sensor9_mean', 'sensor9_std', 'sensor11_mean', 'sensor11_std', 'sensor12_mean', 'sensor12_std', 'sensor13_mean', 'sensor13_std', 'sensor14_mean', 'sensor14_std', 'sensor15_mean', 'sensor15_std', 'sensor17_mean', 'sensor17_std', 'sensor20_mean', 'sensor20_std', 'sensor21_mean', 'sensor21_std', 'RUL']


In [4]:
columns = ['unit_number' , 'time', 'op_settings1' , 'op_settings2' , 'op_settings3'] + [f'sensor{i}' for i in range(1,22)]

original_data = pd.read_csv(base_path / 'data/raw/files/train_FD001.txt', sep = r"\s+", header = None, names = columns) 

print(original_data.columns.tolist())

['unit_number', 'time', 'op_settings1', 'op_settings2', 'op_settings3', 'sensor1', 'sensor2', 'sensor3', 'sensor4', 'sensor5', 'sensor6', 'sensor7', 'sensor8', 'sensor9', 'sensor10', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor16', 'sensor17', 'sensor18', 'sensor19', 'sensor20', 'sensor21']


In [5]:
original_data

,unit_number,time,op_settings1,op_settings2,op_settings3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor12,sensor13,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,519.49,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,519.68,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,520.01,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,519.67,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640


In [6]:
data['unit_number'] = original_data['unit_number']
data['time'] = original_data['time']

data

,op_settings1,op_settings2,op_settings3,sensor2,sensor3,sensor4,sensor7,sensor8,sensor9,sensor11,...,sensor15_std,sensor17_mean,sensor17_std,sensor20_mean,sensor20_std,sensor21_mean,sensor21_std,RUL,unit_number,time
0,0.459770,0.166667,0.0,0.183735,0.406802,0.309757,0.726248,0.242424,0.109755,0.369048,...,0.000000,0.303030,0.000000,0.747899,0.000000,0.753677,0.000000,125,1,1
1,0.609195,0.250000,0.0,0.283133,0.453019,0.352633,0.628019,0.212121,0.100242,0.380952,...,0.132258,0.303030,0.000000,0.711885,0.130435,0.758565,0.017365,125,1,2
2,0.252874,0.750000,0.0,0.343373,0.369523,0.370527,0.710145,0.272727,0.140043,0.250000,...,0.116172,0.202020,0.408248,0.679872,0.169323,0.703945,0.237961,125,1,3
3,0.540230,0.500000,0.0,0.343373,0.256159,0.331195,0.740741,0.318182,0.124518,0.166667,...,0.427568,0.227273,0.353553,0.642857,0.234642,0.692415,0.202745,125,1,4
4,0.390805,0.333333,0.0,0.349398,0.257467,0.404625,0.668277,0.242424,0.149960,0.255952,...,0.394651,0.272727,0.387298,0.625450,0.226338,0.698461,0.178837,125,1,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,0.477011,0.250000,0.0,0.686747,0.587312,0.782917,0.254428,0.439394,0.196491,0.726190,...,0.351904,0.856061,0.401887,0.064226,0.377914,0.120218,0.406119,4,100,196
20627,0.408046,0.083333,0.0,0.701807,0.729453,0.866475,0.162641,0.500000,0.194651,0.708333,...,0.352050,0.856061,0.401887,0.046218,0.390729,0.133416,0.393656,3,100,197
20628,0.522989,0.500000,0.0,0.665663,0.684979,0.775321,0.175523,0.515152,0.198196,0.738095,...,0.396068,0.878788,0.438298,0.039616,0.387600,0.102397,0.436494,2,100,198
20629,0.436782,0.750000,0.0,0.608434,0.746021,0.747468,0.133655,0.530303,0.233285,0.916667,...,0.406169,0.871212,0.442407,0.019808,0.390448,0.091293,0.434224,1,100,199


In [7]:
print(data.columns.tolist())

['op_settings1', 'op_settings2', 'op_settings3', 'sensor2', 'sensor3', 'sensor4', 'sensor7', 'sensor8', 'sensor9', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor17', 'sensor20', 'sensor21', 'raw_RUL', 'sensor2_mean', 'sensor2_std', 'sensor3_mean', 'sensor3_std', 'sensor4_mean', 'sensor4_std', 'sensor7_mean', 'sensor7_std', 'sensor8_mean', 'sensor8_std', 'sensor9_mean', 'sensor9_std', 'sensor11_mean', 'sensor11_std', 'sensor12_mean', 'sensor12_std', 'sensor13_mean', 'sensor13_std', 'sensor14_mean', 'sensor14_std', 'sensor15_mean', 'sensor15_std', 'sensor17_mean', 'sensor17_std', 'sensor20_mean', 'sensor20_std', 'sensor21_mean', 'sensor21_std', 'RUL', 'unit_number', 'time']


In [8]:
exclude_cols = ['unit_number', 'time', 'RUL', 'raw_RUL', 'anomaly_score']

X = data.drop(columns=exclude_cols, errors='ignore')
y = data['RUL']

In [15]:
print(X.columns.tolist())

['op_settings1', 'op_settings2', 'op_settings3', 'sensor2', 'sensor3', 'sensor4', 'sensor7', 'sensor8', 'sensor9', 'sensor11', 'sensor12', 'sensor13', 'sensor14', 'sensor15', 'sensor17', 'sensor20', 'sensor21', 'sensor2_mean', 'sensor2_std', 'sensor3_mean', 'sensor3_std', 'sensor4_mean', 'sensor4_std', 'sensor7_mean', 'sensor7_std', 'sensor8_mean', 'sensor8_std', 'sensor9_mean', 'sensor9_std', 'sensor11_mean', 'sensor11_std', 'sensor12_mean', 'sensor12_std', 'sensor13_mean', 'sensor13_std', 'sensor14_mean', 'sensor14_std', 'sensor15_mean', 'sensor15_std', 'sensor17_mean', 'sensor17_std', 'sensor20_mean', 'sensor20_std', 'sensor21_mean', 'sensor21_std']


In [9]:
y

0        125
1        125
2        125
3        125
4        125
        ... 
20626      4
20627      3
20628      2
20629      1
20630      0
Name: RUL, Length: 20631, dtype: int64

In [10]:
X

,op_settings1,op_settings2,op_settings3,sensor2,sensor3,sensor4,sensor7,sensor8,sensor9,sensor11,...,sensor14_mean,sensor14_std,sensor15_mean,sensor15_std,sensor17_mean,sensor17_std,sensor20_mean,sensor20_std,sensor21_mean,sensor21_std
0,0.459770,0.166667,0.0,0.183735,0.406802,0.309757,0.726248,0.242424,0.109755,0.369048,...,0.207823,0.000000,0.463943,0.000000,0.303030,0.000000,0.747899,0.000000,0.753677,0.000000
1,0.609195,0.250000,0.0,0.283133,0.453019,0.352633,0.628019,0.212121,0.100242,0.380952,...,0.183722,0.257354,0.495930,0.132258,0.303030,0.000000,0.711885,0.130435,0.758565,0.017365
2,0.252874,0.750000,0.0,0.343373,0.369523,0.370527,0.710145,0.272727,0.140043,0.250000,...,0.179610,0.189759,0.482320,0.116172,0.202020,0.408248,0.679872,0.169323,0.703945,0.237961
3,0.540230,0.500000,0.0,0.343373,0.256159,0.331195,0.740741,0.318182,0.124518,0.166667,...,0.178567,0.155735,0.411021,0.427568,0.227273,0.353553,0.642857,0.234642,0.692415,0.202745
4,0.390805,0.333333,0.0,0.349398,0.257467,0.404625,0.668277,0.242424,0.149960,0.255952,...,0.177901,0.135338,0.431904,0.394651,0.272727,0.387298,0.625450,0.226338,0.698461,0.178837
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20626,0.477011,0.250000,0.0,0.686747,0.587312,0.782917,0.254428,0.439394,0.196491,0.726190,...,0.224292,0.101154,0.916834,0.351904,0.856061,0.401887,0.064226,0.377914,0.120218,0.406119
20627,0.408046,0.083333,0.0,0.701807,0.729453,0.866475,0.162641,0.500000,0.194651,0.708333,...,0.222111,0.110913,0.916990,0.352050,0.856061,0.401887,0.046218,0.390729,0.133416,0.393656
20628,0.522989,0.500000,0.0,0.665663,0.684979,0.775321,0.175523,0.515152,0.198196,0.738095,...,0.221841,0.110182,0.937014,0.396068,0.878788,0.438298,0.039616,0.387600,0.102397,0.436494
20629,0.436782,0.750000,0.0,0.608434,0.746021,0.747468,0.133655,0.530303,0.233285,0.916667,...,0.222081,0.108601,0.941461,0.406169,0.871212,0.442407,0.019808,0.390448,0.091293,0.434224


In [11]:
train_mask = data['unit_number'] <= 80
val_mask = data['unit_number'] > 80

X_train = X[train_mask]
y_train = y[train_mask]
X_val = X[val_mask]
y_val = y[val_mask]

In [12]:
model = XGBRegressor(
    n_estimators=150,
    max_depth=5,
    learning_rate=0.05,
    random_state=42,
    early_stopping_rounds=15, 
    eval_metric='rmse'  
)

In [13]:
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",15
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [14]:
model_path = base_path / "models/rul_regressor.joblib"
joblib.dump(model, model_path)

['C:\\Users\\acer\\Desktop\\Vangaurd\\models\\rul_regressor.joblib']